> **BLDC copy (voltage-scaled comparison).** This is `dana-mpc-copy-with-estimator/ControllerAnalysis_Estimator.ipynb` with the robot's own motor, the 5010 360 KV BLDC as a DC-equivalent motor (`robot_constants.py`), its driver current limits, and the controller and test voltages rescaled for it. Everything else, the COG included, is unchanged, so the results compare with the Pololu copy. Every change is marked `BLDC:` and listed in `README.md` in this folder.

# This document Combines and compared Controller and Robot Classes 





## Comparison Simulation Types

1. Single predefined motion path
    * Slow and Fast path duration comparisons 
    * Reducing time duration analysis


In [ ]:
import time

class CodeTimer:
    """
    A versatile timer class for measuring elapsed execution time with 
    support for starting, pausing, resuming, and resetting.
    """
    def __init__(self):
        self.reset()

    def start(self):
        """Starts the timer from a fresh state or restarts it."""
        self._start_time = time.perf_counter()
        self._elapsed_time = 0.0
        self._is_running = True

    def pause(self):
        """Pauses the timer if it is currently running."""
        if self._is_running:
            self._elapsed_time += time.perf_counter() - self._start_time
            self._is_running = False

    def resume(self):
        """Resumes the timer if it was paused."""
        if not self._is_running:
            self._start_time = time.perf_counter()
            self._is_running = True

    def reset(self):
        """Resets all timer counters and stops the timer."""
        self._start_time = 0.0
        self._elapsed_time = 0.0
        self._is_running = False

    @property
    def elapsed(self) -> float:
        """Returns the total elapsed time in seconds, updating live if running."""
        if self._is_running:
            return self._elapsed_time + (time.perf_counter() - self._start_time)
        return self._elapsed_time


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# =======================================================================
# 0. BRING IN FUNCTIONS FROM OTHER NOTEBOOKS
# =======================================================================

from IPython.utils.capture import capture_output
global simulation_parameters, PathGenerator, OmniRobotSimulationExplicit
# This forces the native Jupyter runner to execute the notebook silently
with capture_output():
    %run "PlantModels.ipynb"
    %run "ControllerModels_Estimator.ipynb"
    %run "PathGeneration.ipynb"


## Simulation Runner Function 

## ADDED: sensing assumption

**For now the controller gets the robot's current pose every step, with no delay**, as if a 1 ms estimator (a Kalman filter on its own core) supplied it. Wheel speeds and currents are read every 1 ms as well.

On the robot (firmware reference, not a requirement):
* the camera gives the pose at about 60 Hz, about **50 ms late**;
* the wheel encoders, motor currents and IMU are read every 1 ms;
* the controller loop runs about every 10 ms (here the MPC stays at 20 ms).

Using the late camera properly needs the estimator to correct the pose at the photo's time and replay the 1 ms sensor data up to now. That is a later step.


In [ ]:
# paramaters and classed are defined in xxxx.ipynb and imported above, so we need to declare them as global to avoid "undefined variable" errors
global model_choice, simulation_parameters, PathGenerator
global OmniRobotSimulationExplicit, OmniRobotStateExplicit, OmniRobotSimulationODESolve, OmniRobotStateODESolve
global CascadedPIDController, MPCControllerSimple



def execute_simulation(plant_parm:dict, controller_type: str, ref_path:dict) -> dict:
    """Executes closed-loop simulation for either PID or MPC over duration T."""
    
    if model_choice == 1:
        plant = OmniRobotSimulationExplicit(dict(plant_parm, lugre_deflection=100e-6), random_seed=42)   # ADDED: 100 um bristle (2 um is too stiff for the explicit plant)
        plant_state = OmniRobotStateExplicit()  
        observable_state = plant.get_observable_state(plant_state)  
    elif model_choice == 2:   
        plant = OmniRobotSimulationODESolve(plant_parm, random_seed=42)
        plant_state = OmniRobotStateODESolve()
        observable_state = plant.get_observable_state(plant_state)
    
    # =======================================================================
    # ADDED: MPC weights from Bryson's rule, weight = 1 / (largest acceptable value)^2
    # (Dana's were q_pose=(100, 100, 60), q_vel=(15, 15, 8), r_voltage=0.015, r_slew=0.008)
    # =======================================================================
    wheelbase_radius = plant_parm['Rb']  # wheel distance from the centre: turns a heading error into a distance at the wheels
    max_pos_err = 0.05                   # acceptable position error (m)
    max_vel_err = 0.10                   # acceptable velocity error (m/s)
    max_dV = 0.5 * plant_parm['R'] / 2.18    # BLDC: the same acceptable current change as 0.5 V on the Pololu (0.23 A), V = R i
    max_V = 6.0 * plant_parm['R'] / 2.18     # BLDC: the same acceptable current as 6 V on the Pololu (2.75 A), V = R i.
                                         # NOTE: to be replaced by a current controller (../current_loop_plan).
                                         # acceptable voltage (V): a light cost on the voltage itself. It keeps the MPC from
                                         # driving the wheels into slip, which its model (no friction) cannot predict.
                                         # None = no cost on the voltage, only on its change.

    q_pose_bryson = (1 / max_pos_err**2, 1 / max_pos_err**2, 1 / (max_pos_err / wheelbase_radius)**2)      # x, y, heading
    q_vel_bryson = (1 / max_vel_err**2, 1 / max_vel_err**2, 1 / (max_vel_err / wheelbase_radius)**2)       # x_dot, y_dot, yaw rate
    r_slew_bryson = 1 / max_dV**2                                                                          # cost on (V_j - V_j-1)^2
    r_voltage_bryson = 0.0 if max_V is None else 1 / max_V**2                                              # cost on V_j^2

    if controller_type.lower() == 'pid':
        ctrl = CascadedPIDController(
            plant_parm,
            pos_pid_gains={'kp': 1.5, 'ki': 0.1, 'kd': 0.05},
            wheel_pid_gains={'kp': 0.15 * plant_parm['R'] / 2.18, 'ki': 0.8 * plant_parm['R'] / 2.18, 'kd': 0.0},   # BLDC: the same currents per speed error
            dt=0.02, a_max=3.5, alpha_max=7.0, 
            k_theta=3.0, voltage_limit=13.44     # BLDC: the supply after the PWM limit
        )
    elif controller_type.lower() == 'mpc_1ms':
        # ADDED: Dana's MPC whose velocity comes from her PoseVelocityEstimator run every plant step (1 ms)
        ctrl = MPCControllerSimple1msEstimator(
            plant_parm, horizon_K=8, dt=0.02,
            q_pose=q_pose_bryson, q_vel=q_vel_bryson,
            r_voltage=r_voltage_bryson, r_slew=r_slew_bryson,
            voltage_limit=13.44, current_limit = 17.3, delta_v_max=5.0 * plant_parm['R'] / 2.18   # BLDC: supply, driver limit, same currents
            )
    elif controller_type.lower() == 'mpc_no_vel':
        # ADDED: the same MPC with no velocity term in the stage cost (Dana's idea), q_vel = 0
        ctrl = MPCControllerSimple(
            plant_parm, horizon_K=8, dt=0.02,
            q_pose=q_pose_bryson, q_vel=(0.0, 0.0, 0.0),
            r_voltage=r_voltage_bryson, r_slew=r_slew_bryson,
            voltage_limit=13.44, current_limit = 17.3, delta_v_max=5.0 * plant_parm['R'] / 2.18   # BLDC: supply, driver limit, same currents
            )
    else:
        ctrl = MPCControllerSimple(
            plant_parm, horizon_K=8, dt=0.02,
            q_pose=q_pose_bryson, q_vel=q_vel_bryson,                  # ADDED: Bryson weights above
            r_voltage=r_voltage_bryson, r_slew=r_slew_bryson,          # ADDED: Bryson weights above
            voltage_limit=13.44, current_limit = 17.3, delta_v_max=5.0 * plant_parm['R'] / 2.18   # BLDC: supply, driver limit, same currents
            )

    # Extract reference trajectory data
    t_ref = ref_path['t']
    pos_ref = ref_path['pos']
    vel_ref = ref_path['vel']
    N = len(t_ref)
    duration = t_ref[-1] - t_ref[0]
    
    # Precompute local reference velocity vectors (logged only)
    # GLOBAL FRAME: with the plant's rotation. The MPC is given the GLOBAL reference vel_ref (its velocity state is global).
    V_local_ref = np.zeros_like(vel_ref)
    for i in range(N):
        V_local_ref[i] = OmniRobotSimulationODESolve._rotation(pos_ref[i, 2]).T @ vel_ref[i]

    # Initialize logging arrays   
    log_pos = []
    log_v = []
    log_V = []
    log_vs = []
    log_opt = []
    log_I = []         # ADDED: motor currents at the end of each controller step
    log_V_1ms = []     # ADDED: applied motor voltages, every plant step (1 ms)
    log_I_1ms = []     # ADDED: motor currents, every plant step (1 ms): the current changes within 1-2 ms of a voltage step
    
    dt_ctrl = ctrl.dt
    substeps = int(round(dt_ctrl / plant_parm['dt'])) # holding substeps

    timer = CodeTimer()
    timer.start()
    for k in range(N):
        # ADDED NOTE: the controller gets the robot's current pose with no delay, as if a 1 ms estimator supplied it.
        # On the robot the camera is ~60 Hz and ~50 ms late; a Kalman filter replaying the 1 ms sensor data would be needed.
        observable_state = plant.get_observable_state(plant_state)
        
        p_d = pos_ref[k]
        v_d = vel_ref[k]
        
        if controller_type.lower() == 'pid':
            u = ctrl.compute(p_d, observable_state)
            is_opt = True
        else:
            if k + ctrl.K + 1 <= N:
                p_h = pos_ref[k : k + ctrl.K + 1]
                v_h = vel_ref[k : k + ctrl.K + 1]            # GLOBAL FRAME: global velocity reference
            else:
                pad = (k + ctrl.K + 1) - N
                p_h = np.vstack([pos_ref[k:], np.tile(pos_ref[-1], (pad, 1))])
                v_h = np.vstack([vel_ref[k:], np.zeros((pad, 3))])   # GLOBAL FRAME: global velocity reference
            u, info = ctrl.compute_optimal_u0(measured_State=observable_state, path_ref_horizon=p_h, vel_ref_horizon=v_h, return_info=True)
            is_opt = info['optimal']

        #loop through the substeps to integrate the plant dynamics
        timer.pause()
        for _ in range(substeps):
            plant_state = plant.step(plant_state, u)
            log_V_1ms.append(plant_state.V_actual.copy())     # ADDED
            log_I_1ms.append(plant_state.I.copy())            # ADDED
            if controller_type.lower() == 'mpc_1ms':
                # ADDED: the 1 ms estimator sees every plant step (on the robot: its own core; not timed here)
                ctrl.estimator_update(plant.get_observable_state(plant_state))

        
        # calculate data from plant_state and observable_state
        v_contact = plant.H @ plant_state.v_local
        v_s = v_contact - plant.p['r'] * plant_state.omega

        # log data for analysis  
        log_pos.append(plant_state.pos_global.copy())
        log_v.append(plant_state.v_global.copy())
        log_V.append(u.copy())
        log_vs.append(v_s.copy())
        log_opt.append(is_opt)
        log_I.append(plant_state.I.copy())                    # ADDED
        timer.resume()

    timer.pause()    
    elapsed = timer.elapsed
    log_pos = np.array(log_pos)
    log_v = np.array(log_v)
    log_V = np.array(log_V)
    log_vs = np.array(log_vs)
    log_opt = np.array(log_opt)
    log_I = np.array(log_I)                                   # ADDED
    log_V_1ms = np.array(log_V_1ms)                           # ADDED
    log_I_1ms = np.array(log_I_1ms)                           # ADDED
    t_1ms = t_ref[0] + plant_parm['dt'] * np.arange(1, len(log_I_1ms) + 1)   # ADDED: the time of each plant step
    
    dist_err = np.hypot(pos_ref[:, 0] - log_pos[:, 0], pos_ref[:, 1] - log_pos[:, 1])
    th_err = np.abs((pos_ref[:, 2] - log_pos[:, 2] + np.pi) % (2.0 * np.pi) - np.pi)
    
    return {
        'type': controller_type.upper(), 'duration': duration,
        't': t_ref, 'pos_ref': pos_ref, 'vel_ref': vel_ref, 'V_local_ref': V_local_ref,
        'pos': log_pos, 'v': log_v, 'V': log_V, 'vs': log_vs,
        'dist_err': dist_err, 'th_err': th_err, 'log_opt': log_opt,
        'elapsed': elapsed,
        'I': log_I, 't_1ms': t_1ms, 'V_1ms': log_V_1ms, 'I_1ms': log_I_1ms   # ADDED: currents, and 1 ms voltages and currents
    }




In [ ]:
# paramaters and classed are defined in xxxx.ipynb and imported above, so we need to declare them as global to avoid "undefined variable" errors
global model_choice, simulation_parameters, PathGenerator
global OmniRobotSimulationExplicit, OmniRobotStateExplicit, OmniRobotSimulationODESolve, OmniRobotStateODESolve
global CascadedPIDController, MPCControllerSimple


# =======================================================================
# 1. SETUP SIMULATION PLANT & CONTROLLERS
# =======================================================================

path_time_fast = 8.0
path_time_slow = 4.0
sim_params = simulation_parameters.copy()

pg_comp = PathGenerator(dt=0.02)
start_pose = np.array([0.0, 0.0, 0.0])
target_pose = np.array([3.0, 3.0, np.pi / 4.0])
ref_path_slow = pg_comp.generate_coupled_segment(start_pose, target_pose, duration=path_time_fast)
ref_path_fast = pg_comp.generate_coupled_segment(start_pose, target_pose, duration=path_time_slow)


print("=======================================================================")
print("  🚀 EXECUTING COMPARATIVE BENCHMARK (CASCADED PID vs. CONVEX MPC)     ")
print("=======================================================================")
print(f" [1/4] Running Cascaded PID under Nominal Motion (T = {path_time_slow}s)...")
res_pid_nom = execute_simulation(sim_params,'pid',ref_path_slow)

print(f" [2/4] Running Convex MPC under Nominal Motion (T = {path_time_slow}s)...")
res_mpc_nom = execute_simulation(sim_params,'mpc',ref_path_slow)

print(f" [3/4] Running Cascaded PID under Fast Motion (T = {path_time_fast} s)...")
res_pid_fast = execute_simulation(sim_params,'pid',ref_path_fast)

print(f" [4/4] Running Convex MPC under Fast Motion (T = {path_time_fast} s)...")
res_mpc_fast = execute_simulation(sim_params,'mpc',ref_path_fast)

print("All simulations completed successfully!")

# =======================================================================
# 2. PRINT COMPREHENSIVE PERFORMANCE METRICS TABLE
# =======================================================================
print("\n" + "="*88)
print("             🏆 CASCADED PID vs. CONVEX MPC PERFORMANCE SUMMARY        ")
print("="*88)
print(f"{'Metric':<32} | {'PID (T=8.0s)':<12} | {'MPC (T=8.0s)':<12} | {'PID (T=4.0s)':<12} | {'MPC (T=4.0s)':<12}")
print("-" * 88)
print(f"{'Final Position Error':<32} | {res_pid_nom['dist_err'][-1]*100:6.2f} cm     | {res_mpc_nom['dist_err'][-1]*100:6.2f} cm     | {res_pid_fast['dist_err'][-1]*100:6.2f} cm     | {res_mpc_fast['dist_err'][-1]*100:6.2f} cm")
print(f"{'Final Heading Error':<32} | {np.degrees(res_pid_nom['th_err'][-1]):6.2f}°      | {np.degrees(res_mpc_nom['th_err'][-1]):6.2f}°      | {np.degrees(res_pid_fast['th_err'][-1]):6.2f}°      | {np.degrees(res_mpc_fast['th_err'][-1]):6.2f}°")
print(f"{'Maximum Position Error':<32} | {np.max(res_pid_nom['dist_err'])*100:6.2f} cm     | {np.max(res_mpc_nom['dist_err'])*100:6.2f} cm     | {np.max(res_pid_fast['dist_err'])*100:6.2f} cm     | {np.max(res_mpc_fast['dist_err'])*100:6.2f} cm")
print(f"{'RMS Position Error':<32} | {np.sqrt(np.mean(res_pid_nom['dist_err']**2))*100:6.2f} cm     | {np.sqrt(np.mean(res_mpc_nom['dist_err']**2))*100:6.2f} cm     | {np.sqrt(np.mean(res_pid_fast['dist_err']**2))*100:6.2f} cm     | {np.sqrt(np.mean(res_mpc_fast['dist_err']**2))*100:6.2f} cm")
print(f"{'Peak Commanded Voltage':<32} | {np.max(np.abs(res_pid_nom['V'])):6.1f} V      | {np.max(np.abs(res_mpc_nom['V'])):6.1f} V      | {np.max(np.abs(res_pid_fast['V'])):6.1f} V      | {np.max(np.abs(res_mpc_fast['V'])):6.1f} V")
print(f"{'Peak Contact Slip Velocity':<32} | {np.max(np.abs(res_pid_nom['vs']))*1000:6.1f} mm/s   | {np.max(np.abs(res_mpc_nom['vs']))*1000:6.1f} mm/s   | {np.max(np.abs(res_pid_fast['vs']))*1000:6.1f} mm/s   | {np.max(np.abs(res_mpc_fast['vs']))*1000:6.1f} mm/s")
print(f"{'Optimal Solution Rate':<32} | {'100.0%':<12} | {'100.0%':<12} | {'100.0%':<12} | {'100.0%':<12}")
print(f"{'Execution Time per Step':<32} | {res_pid_nom['elapsed']/len(res_pid_nom['t'])*1000:6.2f} ms    | {res_mpc_nom['elapsed']/len(res_mpc_nom['t'])*1000:6.2f} ms    | {res_pid_fast['elapsed']/len(res_pid_fast['t'])*1000:6.2f} ms    | {res_mpc_fast['elapsed']/len(res_mpc_fast['t'])*1000:6.2f} ms")
print("="*88)

# =======================================================================
# 3. FIGURE 1: SIDE-BY-SIDE 2D TRAJECTORY TRACKING
# =======================================================================
fig, axs = plt.subplots(2, 2, figsize=(14, 13))

def plot_trajectory_panel(ax, res, title, path_color):
    arena = plt.Rectangle((-4.5, -4.5), 9.0, 9.0, fill=False, edgecolor='black', linestyle='--', lw=1.5, label='9m × 9m Arena')
    ax.add_patch(arena)
    
    pos_ref = res['pos_ref']
    pos = res['pos']
    
    ax.plot(pos_ref[:, 0], pos_ref[:, 1], 'k:', lw=1.8, alpha=0.7, label='Reference Setpoint')
    ax.plot(pos[:, 0], pos[:, 1], color=path_color, lw=2.5, label='Actual Robot Path')
    
    ax.plot(pos_ref[0, 0], pos_ref[0, 1], 'gs', markersize=9, label='Start: (0, 0, 0)')
    ax.plot(pos_ref[-1, 0], pos_ref[-1, 1], 'r*', markersize=13, label=r'Target: (3, 3, $\pi/4$)')
    
    # Overlay heading tangent directors (+y_r forward)
    step_jump = max(1, len(pos) // 12)
    indices = np.arange(0, len(pos), step_jump)
    for idx in indices:
        xi, yi, thi = pos[idx, 0], pos[idx, 1], pos[idx, 2]
        u_fwd = -np.sin(thi) * 0.32
        v_fwd =  np.cos(thi) * 0.32
        ax.annotate('', xy=(xi + u_fwd, yi + v_fwd), xytext=(xi, yi),
                    arrowprops=dict(arrowstyle="->", color="crimson", lw=1.5))
                    
    ax.set_xlim(-0.5, 3.8)
    ax.set_ylim(-0.5, 3.8)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Global X Position (m)', fontsize=10)
    ax.set_ylabel('Global Y Position (m)', fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='lower right', fontsize=8.5)

plot_trajectory_panel(axs[0, 0], res_pid_nom, 'Cascaded PID — Nominal Motion (T = 8.0 s)', 'royalblue')
plot_trajectory_panel(axs[0, 1], res_mpc_nom, 'Convex MPC — Nominal Motion (T = 8.0 s)', 'darkorange')
plot_trajectory_panel(axs[1, 0], res_pid_fast, 'Cascaded PID — Fast Motion (T = 4.0 s)', 'royalblue')
plot_trajectory_panel(axs[1, 1], res_mpc_fast, 'Convex MPC — Fast Motion (T = 4.0 s)', 'darkorange')

plt.tight_layout()
plt.show()

# =======================================================================
# 4. FIGURE 2: POSITION & HEADING TRACKING ERRORS
# =======================================================================
fig, axs = plt.subplots(2, 2, figsize=(14, 9))

# Top-Left: Nominal Position Error
axs[0, 0].plot(res_pid_nom['t'], res_pid_nom['dist_err'] * 100.0, 'royalblue', lw=2.0, label=f'Cascaded PID (Peak: {np.max(res_pid_nom["dist_err"])*100:.1f} cm, End: {res_pid_nom["dist_err"][-1]*100:.2f} cm)')
axs[0, 0].plot(res_mpc_nom['t'], res_mpc_nom['dist_err'] * 100.0, 'darkorange', lw=2.2, label=f'Convex MPC (Peak: {np.max(res_mpc_nom["dist_err"])*100:.1f} cm, End: {res_mpc_nom["dist_err"][-1]*100:.2f} cm)')
axs[0, 0].set_title('Nominal Motion (T = 8.0 s) — Euclidean Position Error', fontsize=11, fontweight='bold')
axs[0, 0].set_ylabel('Position Error (cm)', fontsize=10)
axs[0, 0].grid(True, linestyle=':', alpha=0.6)
axs[0, 0].legend(loc='upper right', fontsize=8.5)

# Top-Right: Fast Position Error
axs[0, 1].plot(res_pid_fast['t'], res_pid_fast['dist_err'] * 100.0, 'royalblue', lw=2.0, label=f'Cascaded PID (Peak: {np.max(res_pid_fast["dist_err"])*100:.1f} cm, End: {res_pid_fast["dist_err"][-1]*100:.2f} cm)')
axs[0, 1].plot(res_mpc_fast['t'], res_mpc_fast['dist_err'] * 100.0, 'darkorange', lw=2.2, label=f'Convex MPC (Peak: {np.max(res_mpc_fast["dist_err"])*100:.1f} cm, End: {res_mpc_fast["dist_err"][-1]*100:.2f} cm)')
axs[0, 1].set_title('Fast Motion (T = 4.0 s) — Euclidean Position Error', fontsize=11, fontweight='bold')
axs[0, 1].set_ylabel('Position Error (cm)', fontsize=10)
axs[0, 1].grid(True, linestyle=':', alpha=0.6)
axs[0, 1].legend(loc='upper right', fontsize=8.5)

# Bottom-Left: Nominal Heading Error
axs[1, 0].plot(res_pid_nom['t'], np.degrees(res_pid_nom['th_err']), 'royalblue', lw=2.0, label=f'Cascaded PID (End: {np.degrees(res_pid_nom["th_err"][-1]):.2f}°)')
axs[1, 0].plot(res_mpc_nom['t'], np.degrees(res_mpc_nom['th_err']), 'darkorange', lw=2.2, label=f'Convex MPC (End: {np.degrees(res_mpc_nom["th_err"][-1]):.2f}°)')
axs[1, 0].set_title('Nominal Motion (T = 8.0 s) — Heading Angle Error', fontsize=11, fontweight='bold')
axs[1, 0].set_xlabel('Simulation Time (s)', fontsize=10)
axs[1, 0].set_ylabel('Heading Error (°)', fontsize=10)
axs[1, 0].grid(True, linestyle=':', alpha=0.6)
axs[1, 0].legend(loc='upper right', fontsize=8.5)

# Bottom-Right: Fast Heading Error
axs[1, 1].plot(res_pid_fast['t'], np.degrees(res_pid_fast['th_err']), 'royalblue', lw=2.0, label=f'Cascaded PID (End: {np.degrees(res_pid_fast["th_err"][-1]):.2f}°)')
axs[1, 1].plot(res_mpc_fast['t'], np.degrees(res_mpc_fast['th_err']), 'darkorange', lw=2.2, label=f'Convex MPC (End: {np.degrees(res_mpc_fast["th_err"][-1]):.2f}°)')
axs[1, 1].set_title('Fast Motion (T = 4.0 s) — Heading Angle Error', fontsize=11, fontweight='bold')
axs[1, 1].set_xlabel('Simulation Time (s)', fontsize=10)
axs[1, 1].set_ylabel('Heading Error (°)', fontsize=10)
axs[1, 1].grid(True, linestyle=':', alpha=0.6)
axs[1, 1].legend(loc='upper right', fontsize=8.5)

plt.tight_layout()
plt.show()

# =======================================================================
# 5. FIGURE 3: ACTUATOR VOLTAGES & CONTACT PATCH WHEEL SLIP
# =======================================================================
fig, axs = plt.subplots(2, 2, figsize=(14, 9))

# Top-Left: Nominal Peak Voltages
axs[0, 0].plot(res_pid_nom['t'], np.max(np.abs(res_pid_nom['V']), axis=1), 'royalblue', lw=2.0, label=f'PID Peak |V| ({np.max(np.abs(res_pid_nom["V"])):.1f} V)')
axs[0, 0].plot(res_mpc_nom['t'], np.max(np.abs(res_mpc_nom['V']), axis=1), 'darkorange', lw=2.0, label=f'MPC Peak |V| ({np.max(np.abs(res_mpc_nom["V"])):.1f} V)')
axs[0, 0].axhline(13.44, color='r', linestyle='--', alpha=0.6, label='Supply Ceiling (±13.44V)')   # BLDC: 0.8 x 16.8 V
axs[0, 0].set_title('Nominal Motion (T = 8.0 s) — Peak Commanded Voltage', fontsize=11, fontweight='bold')
axs[0, 0].set_ylabel('Voltage (V)', fontsize=10)
axs[0, 0].set_ylim(0, 13)
axs[0, 0].grid(True, linestyle=':', alpha=0.6)
axs[0, 0].legend(loc='upper right', fontsize=8.5)

# Top-Right: Fast Peak Voltages
axs[0, 1].plot(res_pid_fast['t'], np.max(np.abs(res_pid_fast['V']), axis=1), 'royalblue', lw=2.0, label=f'PID Peak |V| ({np.max(np.abs(res_pid_fast["V"])):.1f} V)')
axs[0, 1].plot(res_mpc_fast['t'], np.max(np.abs(res_mpc_fast['V']), axis=1), 'darkorange', lw=2.0, label=f'MPC Peak |V| ({np.max(np.abs(res_mpc_fast["V"])):.1f} V)')
axs[0, 1].axhline(13.44, color='r', linestyle='--', alpha=0.6, label='Supply Ceiling (±13.44V)')   # BLDC: 0.8 x 16.8 V
axs[0, 1].set_title('Fast Motion (T = 4.0 s) — Peak Commanded Voltage', fontsize=11, fontweight='bold')
axs[0, 1].set_ylabel('Voltage (V)', fontsize=10)
axs[0, 1].set_ylim(0, 13)
axs[0, 1].grid(True, linestyle=':', alpha=0.6)
axs[0, 1].legend(loc='upper right', fontsize=8.5)

# Bottom-Left: Nominal Peak Slip
axs[1, 0].plot(res_pid_nom['t'], np.max(np.abs(res_pid_nom['vs']), axis=1)*1000.0, 'royalblue', lw=1.8, label=f'PID Peak Slip ({np.max(np.abs(res_pid_nom["vs"]))*1000.0:.1f} mm/s)')
axs[1, 0].plot(res_mpc_nom['t'], np.max(np.abs(res_mpc_nom['vs']), axis=1)*1000.0, 'darkorange', lw=1.8, label=f'MPC Peak Slip ({np.max(np.abs(res_mpc_nom["vs"]))*1000.0:.1f} mm/s)')
axs[1, 0].axhline(5.0, color='r', linestyle='--', alpha=0.7, label='Sliding Limit (5 mm/s)')
axs[1, 0].set_title('Nominal Motion (T = 8.0 s) — Peak Contact Patch Slip Velocity', fontsize=11, fontweight='bold')
axs[1, 0].set_xlabel('Simulation Time (s)', fontsize=10)
axs[1, 0].set_ylabel('Peak Slip |vs| (mm/s)', fontsize=10)
axs[1, 0].grid(True, linestyle=':', alpha=0.6)
axs[1, 0].legend(loc='upper right', fontsize=8.5)

# Bottom-Right: Fast Peak Slip
axs[1, 1].plot(res_pid_fast['t'], np.max(np.abs(res_pid_fast['vs']), axis=1)*1000.0, 'royalblue', lw=1.8, label=f'PID Peak Slip ({np.max(np.abs(res_pid_fast["vs"]))*1000.0:.1f} mm/s)')
axs[1, 1].plot(res_mpc_fast['t'], np.max(np.abs(res_mpc_fast['vs']), axis=1)*1000.0, 'darkorange', lw=1.8, label=f'MPC Peak Slip ({np.max(np.abs(res_mpc_fast["vs"]))*1000.0:.1f} mm/s)')
axs[1, 1].axhline(5.0, color='r', linestyle='--', alpha=0.7, label='Sliding Limit (5 mm/s)')
axs[1, 1].set_title('Fast Motion (T = 4.0 s) — Peak Contact Patch Slip Velocity', fontsize=11, fontweight='bold')
axs[1, 1].set_xlabel('Simulation Time (s)', fontsize=10)
axs[1, 1].set_ylabel('Peak Slip |vs| (mm/s)', fontsize=10)
axs[1, 1].grid(True, linestyle=':', alpha=0.6)
axs[1, 1].legend(loc='upper right', fontsize=8.5)

plt.tight_layout()
plt.show()



## ADDED: PID and MPC on more motions (attainable paths)

Dana's path generator (`PathGeneration.ipynb`), with segment times the robot can actually drive. Each segment is a straight line between two waypoints with a smooth quintic speed profile (it starts and stops at rest), so the paths have corners but the robot is stopped at each one.

**BLDC:** the segment times below were chosen for the Pololu, which tops out at about 47 rad/s at 12 V. The BLDC tops out at about 507 rad/s at 13.44 V, so every path here is well within its speed; the percentages in the table are the Pololu's (about 11 times smaller on the BLDC). The times are kept as they are, to compare with the Pololu copy:

| Path | Call | Segment time | Peak wheel speed | Peak acceleration |
|---|---|---|---|---|
| Random | `generate_random_sequence(4, workspace_9m, 10.0, 42)` (her arena and seed) | 10 s | 75% of the limit | 0.3 m/s² |
| Quadrants | `generate_waypoints_sequence(shared_benchmark_wps, 12.0, 42)` (her waypoints) | 12 s | 72% | 0.3 m/s² |
| Quick hops | `generate_random_sequence(6, small_area, 1.5, 7)`, area ±0.5 m | 1.5 s | 73% | 1.9 m/s² |
| Quick hops, faster | the same with 1.25 s | 1.25 s | 88% | 2.7 m/s² |
| Figure eight | `figure_eight_path(0.4, 3.0)` (defined in the cell; not Dana's generator) | 3 s per lap | 88% | 3.8 m/s² |
| Figure eight, tight | `figure_eight_path(0.3, 2.2)` | 2.2 s per lap | see the table | about 5 m/s² |

The long paths and the hops test tracking: with the Pololu the robot runs out of wheel speed before it runs out of grip on straight moves, so they do not slide. **BLDC:** the limit is force instead (17.3 A per motor, 20 A from the battery), not speed. The figure eights curve the whole way round, so they need sideways acceleration all the time, and they slide.

Each path runs through `execute_simulation` unchanged, with PID and the MPC (Bryson weights). The errors are Dana's `dist_err` (her runner compares the pose after each 20 ms step with the reference before it). "Off path" is the largest distance from the reference path itself, ignoring timing (what the X-Y plot shows). "Wheel speed" is the path's peak wheel speed as a share of the 12 V limit. "Sliding" is the share of steps with any wheel slipping faster than 5 mm/s.

Note: `generate_coupled_segment` calls `compute_minimum_duration` when `duration` is None, but that method is not defined, so every call here passes a duration.


In [ ]:
# ADDED: PID and MPC on attainable paths from Dana's path generator
# defined in the notebooks loaded with %run above; declared global so the editor sees them (as Dana does)
global simulation_parameters, PathGenerator, OmniRobotSimulationODESolve, execute_simulation
test_params = simulation_parameters.copy()
test_paths_gen = PathGenerator(dt=0.02)

# Dana's 9 m x 9 m arena with a 0.5 m margin, and her quadrant waypoints (from her PathGeneration demo cell)
arena_half = 9.0 / 2.0
arena_margin = 0.5
workspace_9m = {
    'x_min': -arena_half + arena_margin, 'x_max': arena_half - arena_margin,
    'y_min': -arena_half + arena_margin, 'y_max': arena_half - arena_margin,
    'theta_min': -np.pi, 'theta_max': np.pi
}
shared_benchmark_wps = [
    np.array([ 0.0,  0.0,  0.0]),                   # start at the origin at rest
    np.array([ 3.2,  2.8,  np.radians(90.0)]),      # quadrant I
    np.array([-2.5,  3.5,  np.radians(180.0)]),     # quadrant II
    np.array([-3.5, -2.8,  np.radians(-90.0)]),     # quadrant III
    np.array([ 2.8, -3.2,  0.0])                    # quadrant IV
]
# A small area for quick hops: short moves, high acceleration, within the wheel speed limit
small_area = {'x_min': -0.5, 'x_max': 0.5, 'y_min': -0.5, 'y_max': 0.5, 'theta_min': -np.pi / 2, 'theta_max': np.pi / 2}



def figure_eight_path(size, period, laps=2, ramp=0.3, swing=0.3, dt=0.02):
    """Figure eight in Dana's path format (t, pos, vel; global frame): size [m], period per lap [s].
    The speed ramps up over about ramp seconds; the heading swings by +-swing rad."""
    t = np.arange(0.0, laps * period + 1.0, dt)
    rate = 2 * np.pi / period
    phase = rate * (t - ramp * (1 - np.exp(-t / ramp)))
    phase_dot = rate * (1 - np.exp(-t / ramp))
    pos = np.column_stack((size * np.sin(phase), 0.5 * size * np.sin(2 * phase), swing * np.sin(phase)))
    vel = np.column_stack((size * np.cos(phase) * phase_dot, size * np.cos(2 * phase) * phase_dot, swing * np.cos(phase) * phase_dot))
    path = {'t': t, 'pos': pos, 'vel': vel}
    return path


test_paths = {
    "Random (10 s segments)": test_paths_gen.generate_random_sequence(num_segments=4, workspace_bounds=workspace_9m, segment_duration=10.0, random_seed=42),
    "Quadrants (12 s segments)": test_paths_gen.generate_waypoints_sequence(shared_benchmark_wps, 12.0, 42),
    "Quick hops (1.5 s)": test_paths_gen.generate_random_sequence(num_segments=6, workspace_bounds=small_area, segment_duration=1.5, random_seed=7),
    "Quick hops (1.25 s)": test_paths_gen.generate_random_sequence(num_segments=6, workspace_bounds=small_area, segment_duration=1.25, random_seed=7),
    "Figure eight 0.4 m (3 s)": figure_eight_path(0.4, 3.0),
    "Figure eight 0.3 m (2.2 s)": figure_eight_path(0.3, 2.2),
}


def peak_wheel_speed_share(path, params):
    """Peak wheel speed the path needs, as a share of the supply limit (V_max / Kb; BLDC: 13.44 V, about 507 rad/s): omega = H_drive R(theta)^T v / r."""
    geometry = OmniRobotSimulationODESolve(params)
    omega_needed = np.array([geometry.H_drive @ OmniRobotSimulationODESolve._rotation(theta).T @ v
                             for theta, v in zip(path['pos'][:, 2], path['vel'])]) / params['r']
    omega_limit = params['V_max'] / params['Kb']
    return np.abs(omega_needed).max() / omega_limit


def largest_distance_off_path(robot_xy, reference_xy):
    """Largest distance from the robot's positions to the reference path (the line through its points), ignoring timing."""
    a = reference_xy[:-1]                        # start of each piece of the reference path
    ab = reference_xy[1:] - a                    # each piece
    ab_len2 = np.maximum(np.sum(ab**2, axis=1), 1e-12)
    distances = np.zeros(len(robot_xy))
    for i, p in enumerate(robot_xy):
        s = np.clip(np.sum((p - a) * ab, axis=1) / ab_len2, 0.0, 1.0)     # closest point on each piece
        distances[i] = np.min(np.hypot(*(a + s[:, None] * ab - p).T))
    return distances.max()


test_results = {}
for path_name, path in test_paths.items():
    for controller in ('pid', 'mpc'):
        test_results[(path_name, controller)] = execute_simulation(test_params, controller, path)

# Table: Dana's metrics
print(f"{'Path':26s} {'Wheel speed':>11s} {'Controller':10s} {'Off path (cm)':>13s} {'Max (cm)':>9s} {'RMS (cm)':>9s} {'Final (cm)':>11s} {'Sliding (%)':>12s} {'ms/step':>8s}")
for path_name, path in test_paths.items():
    wheel_share = peak_wheel_speed_share(path, test_params)
    for controller in ('pid', 'mpc'):
        res = test_results[(path_name, controller)]
        sliding = 100 * np.mean(np.any(np.abs(res['vs']) > test_params['slip_band'], axis=1))
        off_path = largest_distance_off_path(res['pos'][:, :2], res['pos_ref'][:, :2])
        print(f"{path_name:26s} {100 * wheel_share:10.0f}% {controller.upper():10s} {100 * off_path:13.1f} {100 * res['dist_err'].max():9.1f} {100 * np.sqrt(np.mean(res['dist_err']**2)):9.1f} "
              f"{100 * res['dist_err'][-1]:11.1f} {sliding:12.1f} {1000 * res['elapsed'] / len(res['t']):8.2f}")

# Paths
fig, axs = plt.subplots(3, 2, figsize=(14, 19))
for ax, path_name in zip(axs.ravel(), test_paths):
    ref = test_results[(path_name, 'pid')]['pos_ref']
    ax.plot(ref[:, 0], ref[:, 1], 'k:', lw=1.8, label='Reference')
    ax.plot(test_results[(path_name, 'pid')]['pos'][:, 0], test_results[(path_name, 'pid')]['pos'][:, 1], color='royalblue', lw=2.0, label='PID')
    ax.plot(test_results[(path_name, 'mpc')]['pos'][:, 0], test_results[(path_name, 'mpc')]['pos'][:, 1], color='darkorange', lw=2.0, label='MPC')
    ax.set_aspect('equal', 'datalim'); ax.grid(True, linestyle=':', alpha=0.6)
    ax.set_title(path_name); ax.set_xlabel('Global X (m)'); ax.set_ylabel('Global Y (m)'); ax.legend(loc='best')
plt.tight_layout()
plt.show()


## ADDED: Dana's turning test

Dana's suggestion: move from (0, 0, 0) to (3, 3, π/2) while turning, and shorten the time. Also a half turn, (3, 3, π). Each is one `generate_coupled_segment` call (straight line, quintic speed profile, turning the whole way).

Shorter times need more wheel speed (the turn adds to it). The shares below are the Pololu's (12 V, about 47 rad/s). **BLDC:** about 507 rad/s at 13.44 V, so all of these are about 11 times smaller and none is too fast:

| Move | 8 s | 6 s | 5 s | 4 s |
|---|---|---|---|---|
| to (3, 3, π/2) | 59% | 79% | 95% | 118% (too fast) |
| to (3, 3, π) | 75% | 100% (at the limit) | 120% (too fast) | |

Three controllers: PID, the MPC (Bryson weights), and the same MPC with no velocity term in the stage cost (`'mpc_no_vel'`, Dana's idea). "Off path" is the largest distance from the line, ignoring timing; "Max error" and "Final" are Dana's `dist_err`; "Heading" is the largest heading error.


In [ ]:
# ADDED: Dana's turning test: (0, 0, 0) -> (3, 3, pi/2) and (3, 3, pi), shorter and shorter
# defined in the notebooks loaded with %run above; declared global so the editor sees them (as Dana does)
global simulation_parameters, PathGenerator, execute_simulation
turn_params = simulation_parameters.copy()
turn_gen = PathGenerator(dt=0.02)
turn_cases = [(np.pi / 2, 8.0), (np.pi / 2, 6.0), (np.pi / 2, 5.0), (np.pi / 2, 4.0),
              (np.pi, 8.0), (np.pi, 6.0), (np.pi, 5.0)]
turn_controllers = ('pid', 'mpc', 'mpc_no_vel')
turn_colors = {'pid': 'royalblue', 'mpc': 'darkorange', 'mpc_no_vel': 'seagreen'}


def distance_off_line(robot_xy, start_xy, end_xy):
    """Largest distance from the robot's positions to the straight line from start to end (the path, ignoring timing)."""
    direction = (end_xy - start_xy) / np.linalg.norm(end_xy - start_xy)
    normal = np.array([-direction[1], direction[0]])
    return np.abs((robot_xy - start_xy) @ normal).max()


turn_results = {}
print(f"{'Move':16s} {'Time':>5s} {'Controller':11s} {'Off path (cm)':>13s} {'Max error (cm)':>14s} {'Final (cm)':>10s} {'Heading (deg)':>13s} {'Sliding (%)':>11s}")
for theta_end, duration in turn_cases:
    path = turn_gen.generate_coupled_segment(np.zeros(3), np.array([3.0, 3.0, theta_end]), duration=duration)
    for controller in turn_controllers:
        res = execute_simulation(turn_params, controller, path)
        turn_results[(theta_end, duration, controller)] = res
        off_path = distance_off_line(res['pos'][:, :2], np.zeros(2), np.array([3.0, 3.0]))
        sliding = 100 * np.mean(np.any(np.abs(res['vs']) > turn_params['slip_band'], axis=1))
        move = f"to (3, 3, {'pi/2' if theta_end < 2 else 'pi'})"
        print(f"{move:16s} {duration:4.1f}s {controller:11s} {100 * off_path:13.1f} {100 * res['dist_err'].max():14.1f} "
              f"{100 * res['dist_err'][-1]:10.1f} {np.degrees(res['th_err'].max()):13.1f} {sliding:11.1f}")

# Paths
fig, axs = plt.subplots(2, 4, figsize=(20, 10))
for ax, (theta_end, duration) in zip(axs.ravel(), turn_cases):
    ref = turn_results[(theta_end, duration, 'pid')]['pos_ref']
    ax.plot(ref[:, 0], ref[:, 1], 'k:', lw=1.8, label='Reference')
    for controller in turn_controllers:
        pos = turn_results[(theta_end, duration, controller)]['pos']
        ax.plot(pos[:, 0], pos[:, 1], color=turn_colors[controller], lw=2.0, label=controller.upper())
    ax.set_aspect('equal', 'datalim'); ax.grid(True, linestyle=':', alpha=0.6)
    ax.set_title(f"to (3, 3, {'pi/2' if theta_end < 2 else 'pi'}) in {duration:.0f} s"); ax.legend(loc='best', fontsize='small')
axs.ravel()[-1].axis('off')
plt.tight_layout()
plt.show()


## ADDED: the simple MPC with a 1 ms velocity estimate

Dana's MPC gets its velocity from her `PoseVelocityEstimator`, which differences the pose once per 20 ms controller step: the velocity is an average over the last 20 ms (about 10 ms old).

`'mpc_1ms'` is the same MPC (same weights and limits) with her same `PoseVelocityEstimator` run every plant step (1 ms), as it would on its own core on the robot; the MPC reads its latest estimate every 20 ms. Same sensing assumption as above (the pose is current). The estimator is not timed (it would run on another core).


In [ ]:
# ADDED: PID, Dana's MPC (20 ms velocity estimate) and the same MPC with a 1 ms velocity estimate
global simulation_parameters, PathGenerator, execute_simulation
compare_params = simulation_parameters.copy()
compare_gen = PathGenerator(dt=0.02)
compare_controllers = ('pid', 'mpc', 'mpc_1ms')
compare_colors = {'pid': 'royalblue', 'mpc': 'darkorange', 'mpc_1ms': 'crimson'}


def figure_eight(size, period, laps=2, ramp=0.3, swing=0.3, dt=0.02):
    """Figure eight in Dana's path format (t, pos, vel; global frame): size [m], period per lap [s]."""
    t = np.arange(0.0, laps * period + 1.0, dt)
    rate = 2 * np.pi / period
    phase = rate * (t - ramp * (1 - np.exp(-t / ramp)))
    phase_dot = rate * (1 - np.exp(-t / ramp))
    pos = np.column_stack((size * np.sin(phase), 0.5 * size * np.sin(2 * phase), swing * np.sin(phase)))
    vel = np.column_stack((size * np.cos(phase) * phase_dot, size * np.cos(2 * phase) * phase_dot, swing * np.cos(phase) * phase_dot))
    path = {'t': t, 'pos': pos, 'vel': vel}
    return path


small_area = {'x_min': -0.5, 'x_max': 0.5, 'y_min': -0.5, 'y_max': 0.5, 'theta_min': -np.pi / 2, 'theta_max': np.pi / 2}
compare_paths = {
    "Dana 8 s": compare_gen.generate_coupled_segment(np.zeros(3), np.array([3.0, 3.0, np.pi / 4]), duration=8.0),
    "Dana 4 s (too fast)": compare_gen.generate_coupled_segment(np.zeros(3), np.array([3.0, 3.0, np.pi / 4]), duration=4.0),
    "Quick hops (1.25 s)": compare_gen.generate_random_sequence(num_segments=6, workspace_bounds=small_area, segment_duration=1.25, random_seed=7),
    "Figure eight 0.4 m (3 s)": figure_eight(0.4, 3.0),
    "Figure eight 0.3 m (2.2 s)": figure_eight(0.3, 2.2),
    "to (3, 3, pi/2) in 5 s": compare_gen.generate_coupled_segment(np.zeros(3), np.array([3.0, 3.0, np.pi / 2]), duration=5.0),
}

compare_results = {}
print(f"{'Path':28s} {'Controller':10s} {'Max error (cm)':>14s} {'RMS (cm)':>9s} {'Final (cm)':>10s} {'Heading (deg)':>13s} {'Sliding (%)':>11s}")
for path_name, path in compare_paths.items():
    for controller in compare_controllers:
        res = execute_simulation(compare_params, controller, path)
        compare_results[(path_name, controller)] = res
        sliding = 100 * np.mean(np.any(np.abs(res['vs']) > compare_params['slip_band'], axis=1))
        print(f"{path_name:28s} {controller:10s} {100 * res['dist_err'].max():14.1f} {100 * np.sqrt(np.mean(res['dist_err']**2)):9.1f} "
              f"{100 * res['dist_err'][-1]:10.1f} {np.degrees(res['th_err'].max()):13.1f} {sliding:11.1f}")

fig, axs = plt.subplots(2, 3, figsize=(18, 12))
for ax, path_name in zip(axs.ravel(), compare_paths):
    ref = compare_results[(path_name, 'pid')]['pos_ref']
    ax.plot(ref[:, 0], ref[:, 1], 'k:', lw=1.8, label='Reference')
    for controller in compare_controllers:
        pos = compare_results[(path_name, controller)]['pos']
        ax.plot(pos[:, 0], pos[:, 1], color=compare_colors[controller], lw=2.0, label=controller)
    ax.set_aspect('equal', 'datalim'); ax.grid(True, linestyle=':', alpha=0.6)
    ax.set_title(path_name); ax.set_xlabel('Global X (m)'); ax.set_ylabel('Global Y (m)'); ax.legend(loc='best', fontsize='small')
plt.tight_layout()
plt.show()
